# **In Silico Perturbation Using SCENIC+ Results**

# 1. Set up Python environment and load data

In [ ]:
import mudata
import os
import scanpy as sc
import anndata
import matplotlib
import matplotlib.pyplot as plt
import adjustText
import numpy as np
import pandas as pd

In [ ]:
import pickle

In [ ]:
from scenicplus.simulation import (
    train_gene_expression_models,
    simulate_perturbation,
    plot_perturbation_effect_in_embedding)

In [ ]:
%matplotlib inline

In [ ]:
outDir = "/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/velocyto"

In [ ]:
data_dir = "/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus_without_C18/scplus_run/Snakemake"

In [ ]:
scplus_mdata = mudata.read(os.path.join(data_dir, "scplusmdata.h5mu"))

In [ ]:
scplus_mdata

# 2. Access AUC GENE values from direct and extended eRegulons

In [ ]:
eRegulon_gene_AUC = anndata.concat(
    [scplus_mdata["direct_gene_based_AUC"], scplus_mdata["extended_gene_based_AUC"]],
    axis = 1,
)
eRegulon_gene_AUC.obs = scplus_mdata.obs

In [ ]:
eRegulon_gene_AUC

In [ ]:
list(eRegulon_gene_AUC.obs['scRNA_counts:Annotated'].unique())

# 3. Run PCA and plot cell types

In [ ]:
sc.pp.pca(eRegulon_gene_AUC)

In [ ]:
color_dict_line = {
    'Chief': "#1f77b4", #blue
    'Foveolar': "#ff7f0e", #orange
  'Neck'        : "#2ca02c", #green
  'Parietal'    : "#d62728", #red
  'Isthmus_5'   : "#b041ff", #purple
  'Isthmus_12'  : "#8c564b", #brown
  'Chga_Hdc'    : "#e377c2", #pink
  'Dividing'    : "#bcbd22", #yellow-green
  'Isthmus_20'  : "#17becf", #cyan
  'Ghrl'        : "#aec7e8", #light blue
  'D_cells'     : "#f3b78a", #light orange
  'Tuft'        : "#98df8a", #light green
  'Parietal_25' : "#ff9896"}  #light red}

In [ ]:
def plot_mm_line_pca(ax):
    texts = []
    # Plot PCA
    ax.scatter(
        eRegulon_gene_AUC.obsm["X_pca"][:, 0],
        eRegulon_gene_AUC.obsm["X_pca"][:, 1],
        color = [color_dict_line[line] for line in eRegulon_gene_AUC.obs["scRNA_counts:Annotated"]],
        s=5
    )
    # Plot labels
    for line in set(eRegulon_gene_AUC.obs["scRNA_counts:Annotated"]):
        line_bc_idc = np.arange(len(eRegulon_gene_AUC.obs_names))[eRegulon_gene_AUC.obs["scRNA_counts:Annotated"] == line]
        avg_x, avg_y = eRegulon_gene_AUC.obsm["X_pca"][line_bc_idc, 0:2].mean(0)
        texts.append(
            ax.text(
                avg_x,
                avg_y,
                line,
                fontweight = "bold",
                fontsize = 9
            )
        )
    adjustText.adjust_text(texts)

fig, ax = plt.subplots()
plot_mm_line_pca(ax)

In [ ]:
def plot_mm_line_pca(ax):
    texts = []

    # 1. Plot PCA points
    ax.scatter(
        eRegulon_gene_AUC.obsm["X_pca"][:, 0],
        eRegulon_gene_AUC.obsm["X_pca"][:, 1],
        color=[color_dict_line[line] for line in eRegulon_gene_AUC.obs["scRNA_counts:Annotated"]],
        s=5,
        alpha=0.6
    )

    # 2. (Optional) cluster labels on the plot
    for line in set(eRegulon_gene_AUC.obs["scRNA_counts:Annotated"]):
        line_bc_idc = np.where(
            eRegulon_gene_AUC.obs["scRNA_counts:Annotated"] == line
        )[0]

        avg_x, avg_y = eRegulon_gene_AUC.obsm["X_pca"][line_bc_idc, :2].mean(axis=0)

        texts.append(
            ax.text(
                avg_x,
                avg_y,
                line,
                fontsize=0
            )
        )

    # 3. Build legend (dummy scatters)
    for line, color in color_dict_line.items():
        ax.scatter([], [], color=color, label=line)

    ax.legend(
        title="Cell type",
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        frameon=False
    )

    # 4. Adjust text last
    adjustText.adjust_text(texts)

fig, ax = plt.subplots()
plot_mm_line_pca(ax)

# 4. Get link between TF and Genes  
**'gene_to_TF' is a dictionary mapping each gene to the list of transcription factors (TFs) that SCENIC+ predicts regulate that gene**

In [ ]:
gene_tf_direct_extended = pd.concat(
    [
        scplus_mdata.uns["direct_e_regulon_metadata"][["Gene", "TF"]].drop_duplicates(),
        scplus_mdata.uns["extended_e_regulon_metadata"][["Gene", "TF"]].drop_duplicates()
    ]
).drop_duplicates()


In [ ]:
gene_to_TF = gene_tf_direct_extended.groupby("Gene")["TF"].apply(lambda tfs: list(tfs)).to_dict()

In [ ]:
gene_to_TF

In [ ]:
len(gene_to_TF.keys())

In [ ]:
tf_to_genes = gene_tf_direct_extended.groupby("TF")["Gene"].apply(lambda genes: list(genes)).to_dict()

# 5. Train model using all genes

In [ ]:
all_genes = scplus_mdata.uns["direct_e_regulon_metadata"].sort_values("triplet_rank")["Gene"].drop_duplicates()
all_genes

In [ ]:
regressors = train_gene_expression_models(
    df_EXP = scplus_mdata["scRNA_counts"].to_df(),
    gene_to_TF = gene_to_TF,
    genes = None,
)

In [ ]:
regressors

In [ ]:
outDir = "/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/velocyto"

In [ ]:
pickle.dump(
    regressors, # not merged
    open(os.path.join(outDir, "regressors.pkl"), "wb")
)

# **Simulate perturbation for a TF of interest**

In [ ]:
perturbation_over_iter = simulate_perturbation(
    df_EXP = scplus_mdata["scRNA_counts"].to_df(),
    perturbation = {"Esrrg": 0},
    keep_intermediate = True,
    n_iter = 5,
    regressors = regressors
)

In [ ]:
perturbation_over_iter

In [ ]:
perturbation_over_iter.keys()